# **1. Install Required Libraries**

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

GENERATION_MODEL = "gpt-4.1-mini"

print("OpenAI client configured successfully.")
print("Model:", GENERATION_MODEL)


Enter your OpenAI API key: ··········
OpenAI client configured successfully.
Model: gpt-4.1-mini


# **2.1 Python Application Functions**


In [ ]:
def employee_lookup(employee_id):
    employees = {101: {"name": "Madhavan", "department": "AI Research Team"},
                 102: {"name": "Rajesh", "department": "Operations"},
                 103: {"name": "Jagadeeshwari", "department": "AI Engineering Team"}}
    return employees.get(employee_id,{"error": "Employee Not Found"})

def calculate_leave_balance(total_leaves, leaves_taken):
    return {"remaining_leaves": total_leaves - leaves_taken}

def calculate(expression):
    allowed = set("0123456789+-*/(). %")

    if not expression or any(char not in allowed for char in expression):
        return {"error": "Only basic arithmetic expressions are allowed."}
    try:
        return {"result": eval(expression, {"__builtins__": {}}, {})}
    except Exception:
        return {"error": "Invalid arithmetic expression."}

def send_email(to, subject, body):
    return {"status": "simulated","message": f"Email prepared for {to} with subject '{subject}'." }

def book_meeting(title, attendee, time):
    return {"status": "simulated","message": f"Meeting '{title}' prepared with {attendee} at {time}."}

# **2.2 LLM input - expects Tools**

In [ ]:
TOOLS = [
    {
        "type": "function",
        "name": "employee_lookup",
        "description": "Look up an employee's name and department using an employee ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "employee_id": {"type": "integer"}
            },
            "required": ["employee_id"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "calculate_leave_balance",
        "description": "Calculate remaining leave from total and taken leaves.",
        "parameters": {
            "type": "object",
            "properties": {
                "total_leaves": {"type": "integer"},
                "leaves_taken": {"type": "integer"}
            },
            "required": ["total_leaves", "leaves_taken"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "calculate",
        "description": "Perform a basic arithmetic calculation.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"}
            },
            "required": ["expression"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "send_email",
        "description": "Prepare a simulated email.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {"type": "string"},
                "subject": {"type": "string"},
                "body": {"type": "string"}
            },
            "required": ["to", "subject", "body"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "book_meeting",
        "description": "Prepare a simulated meeting booking.",
        "parameters": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "attendee": {"type": "string"},
                "time": {"type": "string"}
            },
            "required": ["title", "attendee", "time"],
            "additionalProperties": False
        }
    }
]

print("Available tools:", [tool["name"] for tool in TOOLS])


Available tools: ['employee_lookup', 'calculate_leave_balance', 'calculate', 'send_email', 'book_meeting']


In [ ]:
TOOL_REGISTRY = {"employee_lookup": employee_lookup,
                 "calculate_leave_balance": calculate_leave_balance,
                 "calculate": calculate,
                 "send_email": send_email,
                 "book_meeting": book_meeting}

# **4. Tool Selection**

The model can select the appropriate tool based on the user's request.

## ***4.1 Without Tools***

In [ ]:
question = "Who is employee 102?"

response = client.responses.create(model=GENERATION_MODEL,
                                   input=question)
print(response.output_text)

I don't see any employee information in the current context. Could you please provide more details or share the document/image that contains the employee data?


## ***4.2 With Tool***

In [ ]:
question = "Who is employee 102?"

response = client.responses.create(model=GENERATION_MODEL,
                                   input=question,
                                   tools=TOOLS)
print("-"*30)
print(response.output)
print("-"*30)

for item in response.output:
    if item.type == "function_call":
        print("Selected tool:", item.name)
        print("Arguments:", item.arguments)

------------------------------
[ResponseFunctionToolCall(arguments='{"employee_id":102}', call_id='call_1rE8vjjnkW7Wn9WIu4d8040R', name='employee_lookup', type='function_call', id='fc_0494de5c12a486f9006aacf541db7887d1a4eb0ac71b870b8b', caller=None, namespace=None, status='completed')]
------------------------------
Selected tool: employee_lookup
Arguments: {"employee_id":102}


# **Level 1: User Prompt Level**
- *the user's request itself is incomplete or invalid.*

### **1. Employee ID with wrong type**

In [ ]:
question = "Find employee AB and tell me their name and department."

response = client.responses.create(model=GENERATION_MODEL,
                                   input=question,
                                   tools=TOOLS)

print("-"*30)
print(response.output)
print("-"*30)

for item in response.output:
    if item.type == "function_call":
        print("Selected tool:", item.name)
        print("Arguments:", item.arguments)
    else:
        print("No tool call was produced.")
        print("Final output:", response.output_text)

------------------------------
[ResponseFunctionToolCall(arguments='{"employee_id":103}', call_id='call_ag3uFglbZtiOYcdDQKMp16sG', name='employee_lookup', type='function_call', id='fc_051fc15020720125006aacf576564887d19838c2b2c7dfa47b', caller=None, namespace=None, status='completed')]
------------------------------
Selected tool: employee_lookup
Arguments: {"employee_id":103}


In [ ]:
def client_response_extractor(question):
  response = client.responses.create(model=GENERATION_MODEL,
                                    input=question,
                                    tools=TOOLS)
  print("-"*30)
  print(response.output)
  print("-"*30)

  for item in response.output:
      if item.type == "function_call":
          print("Selected tool:", item.name)
          print("Arguments:", item.arguments)
      else:
          print("No tool call was produced.")
          print("Final output:", response.output_text)

### **2. Unknown employee ID**

In [ ]:
question = "Find employee 999 and tell me their name and department."
client_response_extractor(question)

------------------------------
[ResponseFunctionToolCall(arguments='{"employee_id":999}', call_id='call_iu5MYAoxR4HbRT4hTsJh8Ac5', name='employee_lookup', type='function_call', id='fc_08a0cbf282984011006aacf5837cc487d1adf7f50e256f2184', caller=None, namespace=None, status='completed')]
------------------------------
Selected tool: employee_lookup
Arguments: {"employee_id":999}


### **3. Missing leave values**

In [ ]:
question = "Calculate the leave balance, but I will not give you total leaves or leaves taken."
client_response_extractor(question)

------------------------------
[ResponseOutputMessage(id='msg_0456bf361d770cab006aacf589736c87d18b97454749811dc3', content=[ResponseOutputText(annotations=[], text='To calculate your leave balance, I will need to know the total number of leaves you are entitled to and the number of leaves you have already taken. Could you please provide these details?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]
------------------------------
No tool call was produced.
Final output: To calculate your leave balance, I will need to know the total number of leaves you are entitled to and the number of leaves you have already taken. Could you please provide these details?


### **4. Division by zero**

In [ ]:
question = "Calculate 10 divided by zero."
client_response_extractor(question)

------------------------------
[ResponseOutputMessage(id='msg_0352423b7cbd4671006aacf59eefa887d19faf8c9f74721e1e', content=[ResponseOutputText(annotations=[], text='Division by zero is undefined in mathematics. It is not possible to divide a number by zero. If you have any other calculation or question, feel free to ask!', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]
------------------------------
No tool call was produced.
Final output: Division by zero is undefined in mathematics. It is not possible to divide a number by zero. If you have any other calculation or question, feel free to ask!


### **5. Unsupported calculation**

In [ ]:
question = "Calculate DROP TABLE employees."
client_response_extractor(question)

------------------------------
[ResponseOutputMessage(id='msg_05f855028b99525a006aacf5bfde6c87d1a6922cb716d4660d', content=[ResponseOutputText(annotations=[], text='It appears you have written a SQL command: "DROP TABLE employees." \n\nIf you intended to ask for an explanation or effect of this command, it will delete the entire table named "employees" from the database, including all its data and structure.\n\nIf you want me to perform a calculation or something else, please clarify. Note that I cannot actually execute SQL commands on databases.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]
------------------------------
No tool call was produced.
Final output: It appears you have written a SQL command: "DROP TABLE employees." 

If you intended to ask for an explanation or effect of this command, it will delete the entire table named "employees" from the database, including all its data and structure.

If you want me to perform

### **6. Invalid email recipient**

In [ ]:
question = "Prepare an email to not-an-email saying hello."
client_response_extractor(question)

------------------------------
[ResponseFunctionToolCall(arguments='{"to":"not-an-email","subject":"Hello","body":"Hello"}', call_id='call_885kA97fpoa0AgFzmLxTRbko', name='send_email', type='function_call', id='fc_01043a4873253c54006aacf5f81a9c87d1af191524d134f143', caller=None, namespace=None, status='completed')]
------------------------------
Selected tool: send_email
Arguments: {"to":"not-an-email","subject":"Hello","body":"Hello"}


### **7. Missing meeting title**

In [ ]:
question = "Book a meeting with John tomorrow, but I have not provided a meeting title."
client_response_extractor(question)

------------------------------
[ResponseOutputMessage(id='msg_027866b5f6e9ddf3006aacf6ab2cdc87d19e4b903683dca1bd', content=[ResponseOutputText(annotations=[], text='Could you please provide the time for the meeting tomorrow and also let me know if you have a preferred title for the meeting or should I use a generic one like "Meeting with John"?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]
------------------------------
No tool call was produced.
Final output: Could you please provide the time for the meeting tomorrow and also let me know if you have a preferred title for the meeting or should I use a generic one like "Meeting with John"?


# ==============================================================================

# **Level 2 & 3: Tool Argument Level & Tool Execution Level Validation**

# **Step 1**: Define the Tool Safety Settings or Tool Safety Wrapper

In [ ]:
import inspect
import json

def execute_tool_safely(tool_name, arguments):
    if tool_name not in TOOL_REGISTRY:
        return {"status": "error","error_type": "UnknownTool","error_message": f"Unknown tool requested: {tool_name}"}

    function = TOOL_REGISTRY[tool_name]

    try:
        # Python signature validation: catches missing and unexpected arguments.
        bound = inspect.signature(function).bind(**arguments)

        # Explicit type validation for demonstration.
        if tool_name == "employee_lookup":
            if not isinstance(arguments.get("employee_id"),int):
                raise TypeError("employee_id must be an integer.")

        elif tool_name == "calculate_leave_balance":
            for name in ("total_leaves", "leaves_taken"):
                value = arguments.get(name)
                if not isinstance(value, int):
                    raise TypeError(f"{name} must be an integer.")

        elif tool_name == "calculate":
            if not isinstance(arguments.get("expression"), str):
                raise TypeError("expression must be a string.")

        elif tool_name == "send_email":
            for name in ("to", "subject", "body"):
                if not isinstance(arguments.get(name), str):
                    raise TypeError(f"{name} must be a string.")

            if "@" not in arguments["to"]:
                raise ValueError("Invalid email address.")

        elif tool_name == "book_meeting":
            for name in ("title", "attendee", "time"):
                if not isinstance(arguments.get(name), str):
                    raise TypeError(f"{name} must be a string.")

            if not arguments["title"].strip():
                raise ValueError("Meeting title cannot be empty.")

        result = function(**bound.arguments)
        return {"status": "success","result": result}

    except TypeError as e:
        return {"status": "error","error_type": "TypeError","error_message": str(e)}

    except ValueError as e:
        return {"status": "error","error_type": "ValueError","error_message": str(e)}

    except Exception as e:
        return {"status": "error","error_type": type(e).__name__,"error_message": str(e)}

print("Safe tool execution wrapper ready.")

Safe tool execution wrapper ready.


### **Negative Test 1: employee_lookup — wrong type**

In [ ]:
result = execute_tool_safely("employee_lookup", {"employee_id": "ABC"})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': 'employee_id must be an integer.'}


### **Negative Test 2: employee_lookup — missing parameter**

In [ ]:
result = execute_tool_safely("employee_lookup", {})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': "missing a required argument: 'employee_id'"}


### **Negative Test 3: employee_lookup — unknown employee**

In [ ]:
result = execute_tool_safely("employee_lookup", {"employee_id": 999})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'success', 'result': {'error': 'Employee Not Found'}}


### **Negative Test 4: calculate_leave_balance — wrong type**

In [ ]:
result = execute_tool_safely("calculate_leave_balance", {"total_leaves": "24", "leaves_taken": 7})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': 'total_leaves must be an integer.'}


In [ ]:
result = execute_tool_safely("calculate_leave_balance", 24)

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': 'inspect.Signature.bind() argument after ** must be a mapping, not int'}


### **Negative Test 5: calculate_leave_balance — missing parameter**

In [ ]:
result = execute_tool_safely("calculate_leave_balance", {"total_leaves": 24})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': "missing a required argument: 'leaves_taken'"}


### **Negative Test 6: calculate_leave_balance — unexpected parameter**

In [ ]:
result = execute_tool_safely("calculate_leave_balance", {"total_leaves": 24, "leaves_taken": 7, "extra": 1})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': "got an unexpected keyword argument 'extra'"}


### **Negative Test 7: calculate_leave_balance — negative leaves taken**

In [ ]:
result = execute_tool_safely("calculate_leave_balance", {"total_leaves": 24, "leaves_taken": -5})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'success', 'result': {'remaining_leaves': 29}}


### **Negative Test 8: calculate — division by zero**

In [ ]:
result = execute_tool_safely("calculate", {"expression": "10 / 0"})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'success', 'result': {'error': 'Invalid arithmetic expression.'}}


### **Negative Test 9: calculate — unsupported expression**

In [ ]:
result = execute_tool_safely("calculate", {"expression": "DROP TABLE employees"})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'success', 'result': {'error': 'Only basic arithmetic expressions are allowed.'}}


### **Negative Test 10: calculate — wrong type**

In [ ]:
result = execute_tool_safely("calculate", {"expression": 125})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': 'expression must be a string.'}


### **Negative Test 11: calculate — empty expression**

In [ ]:
result = execute_tool_safely("calculate", {"expression": ""})

print("Tool Result:")
print(result)

### **Negative Test 12: send_email — invalid email**

In [ ]:
result = execute_tool_safely("send_email", {"to": "not-an-email", "subject": "Test", "body": "Hello"})

print("Tool Result:")
print(result)

### **Negative Test 13: send_email — missing body**

In [ ]:
result = execute_tool_safely("send_email", {"to": "hr@company.com", "subject": "Test"})

print("Tool Result:")
print(result)

### **Negative Test 14: send_email — wrong subject type**

In [ ]:
result = execute_tool_safely("send_email", {"to": "hr@company.com", "subject": 123, "body": "Hello"})

print("Tool Result:")
print(result)

### **Negative Test 15: send_email — unexpected parameter**

In [ ]:
result = execute_tool_safely("send_email", {"to": "hr@company.com", "subject": "Test", "body": "Hello", "cc": "x@company.com"})

print("Tool Result:")
print(result)

### **Negative Test 16: book_meeting — empty title**

In [ ]:
result = execute_tool_safely("book_meeting", {"title": "", "attendee": "John", "time": "10:00 AM"})

print("Tool Result:")
print(result)

### **Negative Test 17: book_meeting — wrong attendee type**

In [ ]:
result = execute_tool_safely("book_meeting", {"title": "Review", "attendee": 123, "time": "10:00 AM"})

print("Tool Result:")
print(result)

### **Negative Test 18: book_meeting — missing time**

In [ ]:
result = execute_tool_safely("book_meeting", {"title": "Review", "attendee": "John"})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': "missing a required argument: 'time'"}


### **Negative Test 19: book_meeting — unexpected parameter**

In [ ]:
result = execute_tool_safely("book_meeting", {"title": "Review", "attendee": "John", "time": "10:00 AM", "location": "Bangalore"})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': "got an unexpected keyword argument 'location'"}


In [ ]:
import json

def run_agent_with_error_handling(user_input, max_steps=5):
    input_items = [{"role": "user", "content": user_input}]

    for step in range(1, max_steps + 1):
        response = client.responses.create(model=GENERATION_MODEL,
                                           input=input_items,
                                           tools=TOOLS )

        tool_calls = [item for item in response.output if item.type == "function_call"]

        if not tool_calls:
            return response.output_text

        input_items.extend([item.model_dump() for item in response.output]) #--> part 1 work is done

        for tool_call in tool_calls:
            try:
                arguments = json.loads(tool_call.arguments)
                tool_result = execute_tool_safely(tool_call.name, arguments)
            except json.JSONDecodeError as e:
                tool_result = {"status": "error","error_type": "InvalidJSON","error_message": f"Could not parse tool arguments: {e}"}

            print(f"Step {step}")
            print("Tool:", tool_call.name)
            print("Arguments:", arguments)
            print("Result:", tool_result)

            input_items.append({
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": json.dumps(tool_result)
            })

    return "Maximum agent steps reached."

#**Debugging Version of Agent**

In [ ]:
import json


def print_input_items(input_items, title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    for i, item in enumerate(input_items):
        print(f"\nINPUT ITEM {i}:")
        print(item)


def run_agent_with_error_handling(user_input, max_steps=5):

    input_items = [
        {
            "role": "user",
            "content": user_input
        }
    ]

    print_input_items(
        input_items,
        "INITIAL INPUT ITEMS"
    )

    for step in range(1, max_steps + 1):

        print("\n" + "#" * 80)
        print(f"STARTING STEP {step}")
        print("#" * 80)

        print_input_items(
            input_items,
            f"STEP {step} - INPUT ITEMS BEFORE LLM"
        )

        response = client.responses.create(
            model=GENERATION_MODEL,
            input=input_items,
            tools=TOOLS
        )

        print("\n" + "-" * 80)
        print(f"STEP {step} - LLM RESPONSE")
        print("-" * 80)

        for item in response.output:
            print(item.model_dump())

        tool_calls = [
            item
            for item in response.output
            if item.type == "function_call"
        ]

        print("\nTool calls found:", len(tool_calls))

        if not tool_calls:
            print("\nNo more tool calls.")
            print("Final answer:", response.output_text)
            return response.output_text

        # Add LLM's function call(s) to history
        input_items.extend(
            [item.model_dump() for item in response.output]
        )

        print_input_items(
            input_items,
            f"STEP {step} - AFTER ADDING LLM RESPONSE"
        )

        # Process every tool call returned by the LLM
        for tool_call in tool_calls:

            print("\n" + "-" * 80)
            print(f"PROCESSING TOOL CALL IN STEP {step}")
            print("-" * 80)

            print("Tool name:", tool_call.name)
            print("Raw arguments:", tool_call.arguments)

            try:
                # Convert JSON string → Python dictionary
                arguments = json.loads(tool_call.arguments)

                print("Parsed arguments:", arguments)

                # Safely execute the requested tool
                tool_result = execute_tool_safely(
                    tool_call.name,
                    arguments
                )

            except json.JSONDecodeError as e:

                tool_result = {
                    "status": "error",
                    "error_type": "InvalidJSON",
                    "error_message": (
                        f"Could not parse tool arguments: {e}"
                    )
                }

                # Useful so printing does not fail if JSON parsing failed
                arguments = "<Invalid JSON>"

            print("\nTool execution result:")
            print(tool_result)

            # Add tool result to conversation history
            input_items.append(
                {
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": json.dumps(tool_result)
                }
            )

            print_input_items(
                input_items,
                f"STEP {step} - AFTER ADDING TOOL RESULT"
            )

    return "Maximum agent steps reached."

### **Single-step task**

In [ ]:
run_agent_with_error_handling("Find employee 102 and tell me their name and department.")

Step 1
Tool: employee_lookup
Arguments: {'employee_id': 102}
Result: {'status': 'success', 'result': {'name': 'Rajesh', 'department': 'Operations'}}


'The employee with ID 102 is named Rajesh and works in the Operations department.'

### **Multi-step task**

In [ ]:
run_agent_with_error_handling("""Find employee 103. Then prepare a simulated email to HR that mentions the employee's name and
                               department and asks HR to confirm the employee's leave balance.""")


INITIAL INPUT ITEMS

INPUT ITEM 0:
{'role': 'user', 'content': "Find employee 103. Then prepare a simulated email to HR that mentions the employee's name and\n                               department and asks HR to confirm the employee's leave balance."}

################################################################################
STARTING STEP 1
################################################################################

STEP 1 - INPUT ITEMS BEFORE LLM

INPUT ITEM 0:
{'role': 'user', 'content': "Find employee 103. Then prepare a simulated email to HR that mentions the employee's name and\n                               department and asks HR to confirm the employee's leave balance."}

--------------------------------------------------------------------------------
STEP 1 - LLM RESPONSE
--------------------------------------------------------------------------------
{'arguments': '{"employee_id":103}', 'call_id': 'call_3wVINPa78WsDD7I43c8DDZ3H', 'name': 'employee_lookup', 't

"I have identified the employee: Jagadeeshwari from the AI Engineering Team. I prepared an email to HR to confirm the employee's leave balance. However, the email address for HR was not specified or invalid. Could you please provide the correct email address for HR so I can send the email?"

# ==============================================================================

# **7. FINAL VALIDATION: LLMs Response Vs AI Agent**

In [ ]:
user_input = 'Find employee 999 and tell me their name and department.'

print("USER INPUT:", user_input)
print("-" * 70)

final_answer = run_agent_with_error_handling(user_input)

print("-" * 70)
print("FINAL ANSWER:", final_answer)

USER INPUT: Find employee 999 and tell me their name and department.
----------------------------------------------------------------------
Step 1
Tool: employee_lookup
Arguments: {'employee_id': 999}
Result: {'status': 'success', 'result': {'error': 'Employee Not Found'}}
----------------------------------------------------------------------
FINAL ANSWER: I couldn't find an employee with the ID 999. Could you please verify the ID or provide another one?


In [ ]:
user_input = 'Calculate 10 / 0.'

print("USER INPUT:", user_input)
print("-" * 70)

final_answer = run_agent_with_error_handling(user_input)

print("-" * 70)
print("FINAL ANSWER:", final_answer)

USER INPUT: Calculate 10 / 0.
----------------------------------------------------------------------
----------------------------------------------------------------------
FINAL ANSWER: Division by zero is undefined in mathematics, so 10 / 0 cannot be calculated. If you need help with another calculation, feel free to ask!


In [ ]:
user_input = 'Calculate DROP TABLE employees.'

print("USER INPUT:", user_input)
print("-" * 70)

final_answer = run_agent_with_error_handling(user_input)

print("-" * 70)
print("FINAL ANSWER:", final_answer)

USER INPUT: Calculate DROP TABLE employees.
----------------------------------------------------------------------
----------------------------------------------------------------------
FINAL ANSWER: It looks like you typed "DROP TABLE employees," which is an SQL command used to delete the entire "employees" table from a database. This is a data definition operation and not a calculation.

If you want me to help with calculations or queries related to the employees' data, please provide more details or clarify your request.


In [ ]:
user_input = 'Prepare an email to not-an-email saying hello.'

print("USER INPUT:", user_input)
print("-" * 70)

final_answer = run_agent_with_error_handling(user_input)

print("-" * 70)
print("FINAL ANSWER:", final_answer)

USER INPUT: Prepare an email to not-an-email saying hello.
----------------------------------------------------------------------
Step 1
Tool: send_email
Arguments: {'to': 'not-an-email', 'subject': 'Greetings', 'body': 'Hello'}
Result: {'status': 'error', 'error_type': 'ValueError', 'error_message': 'Invalid email address.'}
----------------------------------------------------------------------
FINAL ANSWER: The email address "not-an-email" you provided is invalid. Could you please provide a valid email address to send the greeting to?


In [ ]:
user_input = 'Book a meeting with John but leave the title empty.'

print("USER INPUT:", user_input)
print("-" * 70)

final_answer = run_agent_with_error_handling(user_input)

print("-" * 70)
print("FINAL ANSWER:", final_answer)

USER INPUT: Book a meeting with John but leave the title empty.
----------------------------------------------------------------------
----------------------------------------------------------------------
FINAL ANSWER: Could you please provide the time or date for the meeting with John? Also, is there a specific email address or any additional details I should know?


# **THE END.**